# C04 技术指标策略与分散化

本节采用“1 主 1 辅”结构：
- 主策略：Bollinger
- 对照策略：RSI


## 教学意图
- 先讲透一个策略（信号、仓位、风险）
- 再用第二个策略做对照
- 用多资产等权展示分散化效果


In [ ]:
MODE = "offline"
START_DATE = "2021-01-01"
END_DATE = "2022-12-31"
UNIVERSE = ["ASSET_A", "ASSET_B", "ASSET_C", "ASSET_D"]
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


In [ ]:
def make_prices(universe, start_date, end_date):
    dates = pd.bdate_range(start_date, end_date)
    data = {}
    for i, asset in enumerate(universe):
        # 不同资产设置不同波动率，便于演示分散化
        ret = np.random.normal(0.00015 + i * 0.00005, 0.012 + i * 0.001, size=len(dates))
        data[asset] = 100 * np.cumprod(1 + ret)
    return pd.DataFrame(data, index=dates)

price = make_prices(UNIVERSE, START_DATE, END_DATE)
ret = price.pct_change().fillna(0)
price.tail()


## Bollinger 策略规则
- `z = (price - MA20) / STD20`
- 入场：`z < -1`
- 出场：`z > 0`


In [ ]:
def boll_pos(series, n=20, entry=-1.0, exit_level=0.0):
    ma = series.rolling(n).mean()
    sd = series.rolling(n).std()
    z = (series - ma) / (sd + 1e-12)
    out = pd.Series(0.0, index=series.index)
    holding = 0.0

    for i in range(len(series)):
        if np.isnan(z.iloc[i]):
            out.iloc[i] = holding
            continue
        if holding == 0 and z.iloc[i] < entry:
            holding = 1.0
        elif holding == 1 and z.iloc[i] > exit_level:
            holding = 0.0
        out.iloc[i] = holding

    return out

boll_position = pd.DataFrame({c: boll_pos(price[c]) for c in price.columns})
boll_ret = (boll_position.shift(1).fillna(0) * ret).mean(axis=1)
boll_nav = (1 + boll_ret).cumprod()


## RSI 对照策略规则
- RSI14 < 30 买入
- RSI14 > 55 卖出


In [ ]:
def rsi(series, n=14):
    delta = series.diff()
    up = delta.clip(lower=0).rolling(n).mean()
    down = (-delta.clip(upper=0)).rolling(n).mean()
    rs = up / (down + 1e-12)
    return 100 - (100 / (1 + rs))


def rsi_pos(series, n=14, low=30, high=55):
    rr = rsi(series, n)
    out = pd.Series(0.0, index=series.index)
    holding = 0.0

    for i in range(len(series)):
        if np.isnan(rr.iloc[i]):
            out.iloc[i] = holding
            continue
        if holding == 0 and rr.iloc[i] < low:
            holding = 1.0
        elif holding == 1 and rr.iloc[i] > high:
            holding = 0.0
        out.iloc[i] = holding

    return out

rsi_position = pd.DataFrame({c: rsi_pos(price[c]) for c in price.columns})
rsi_ret = (rsi_position.shift(1).fillna(0) * ret).mean(axis=1)
rsi_nav = (1 + rsi_ret).cumprod()


In [ ]:
perf = pd.DataFrame({"Bollinger": boll_nav, "RSI": rsi_nav})
fig, ax = plt.subplots(figsize=(10, 4))
perf.plot(ax=ax, title="C04 策略净值对比")
ax.set_ylabel("NAV")
plt.show()


def metrics(r):
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_ret / (ann_vol + 1e-12)
    mdd = ((1 + r).cumprod() / (1 + r).cumprod().cummax() - 1).min()
    return ann_ret, ann_vol, sharpe, mdd

summary = pd.DataFrame(
    [metrics(boll_ret), metrics(rsi_ret)],
    columns=["annual_return", "annual_vol", "sharpe", "max_drawdown"],
    index=["Bollinger", "RSI"],
).round(4)
summary


## 小结与练习
- 分散化通常改善风险表现，但不保证收益更高。
- 练习：把等权改成波动率倒数加权，比较最大回撤。
